# EMG data classification initial testing 

test space for extracting features

## Libraries 

In [2]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")

## Defining Initial Variables  

In [13]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/full_EEG_dataset" # REPLACE WITH OWN PATH 
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

Pre-Processing Steps

In [6]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

## Feature Extraction 

### Feature extraction function 

In [7]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

     # frequency features (did not use)
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]

  
     return var, rms, wl, fmd


#### Plotting function 

In [39]:
def plot_epoch(corr,zygo,Var_corr,Wl_corr,RMS_corr,
               Var_zygo,Wl_zygo,RMS_zygo):
    fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)

    ax_corr = ax[0].twinx()
    ax_zygo = ax[1].twinx()

    title = fig.suptitle(
        "Facial EMG Response Following Stimulus",
        fontsize=14,
    )

    corrugator_color = "#734F62"
    zygomatic_color = "#6F9359"

    time = np.linspace(0,10,2251)
    trigger_time = time[0]

                    
    # Plot Corr
    #ax[0].axvline(trigger_time, label="Trigger Channel", color="black")
    ax[0].plot(time,corr,  color=corrugator_color)
    ax[0].set_ylim(-100, 100)
    ax[0].set_ylabel("Corr",size=12)
    # ax[0].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))

    # Plot Zygo
    ax[1].plot(time, zygo, lw=1.5, color=zygomatic_color)
    ax[1].set_ylim(-100, 100)
    ax[1].set_ylabel("Zygo",size=12)
    # ax[1].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))


    # plotting features 
    ax_corr.plot(time,Var_corr,  color="magenta", alpha=0.7,)
    ax_corr.plot(time,Wl_corr,  color="blue", alpha=0.7,)
    ax_corr.plot(time,RMS_corr*10,  color="black", alpha=0.7,) # multiply by 10 for scaling 
    ax_corr.set_ylim(-1500, 1500)

    ax_zygo.plot(time,Var_zygo,  color="magenta", label="variance", alpha=0.7,)
    ax_zygo.plot(time,Wl_zygo,  color="blue", alpha=0.7)
    ax_zygo.plot(time,RMS_zygo*10,  color="black", alpha=0.7)
    ax_zygo.set_ylim(-1500, 1500)

    ax_zygo.axis("off")
    ax_corr.axis("off")


    for a in ax:
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

        plt.tight_layout()
        #plt.show()
    return fig 

In [37]:
def on_key(event):
    global current_index, fig, subject_epoch, subject, block

    if event.key == 'right':
        current_index = (current_index + 1) % len(subject_epoch)
    elif event.key == 'left':
        current_index = (current_index - 1) % len(subject_epoch)
    elif event.key == 'escape':
        print("Quitting the plot!")
        plt.close(fig)
        return

    epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
    epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

    epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
    epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

    plt.close(fig)
    fig = plot_epoch(epoch_corr, epoch_zygo, epoch_var_corr, epoch_wl_corr, epoch_rms_corr,
                     epoch_var_zygo, epoch_wl_zygo, epoch_rms_zygo)
    fig.canvas.mpl_connect('key_press_event', on_key)   
    fig.suptitle(f"subject: {subject} | block: {block} | epoch: {current_index}", fontsize=14)
    plt.show(block=False)
    print(current_index)


### Loop through all files 

In [ ]:
#define data frame if want to store all features 
def define_df():
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "True_Muscle_Activated",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "WL_Corr",
            "Var_Corr",
            "RMS_Corr", 
        ]
        ) 
    return features 

In [ ]:
# extracting features for classification 
i = 0 
# features_results_mat = [] if want to fill data frame with features for each subject 

for root,dirs,files in os.walk(raw_path): # loop through file 
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0][:-2]
            block = file.split(".")[0][-2:]
            print(file)
            print(subject+block)


            # need to leave out these subjects 
            if (subject == "NL02IF" or 
                subject == "NL05WW" or 
                subject == "NL01SS" or 
                subject == "RL11JH" or 
                subject == "RL12JL" or 
                subject == "RL07BR"):
                continue
            
            # features = def_df() # define dataframe 
            subject_epoch, _ = pre_process_subjets(subject,block) # preprocess subject to get epochs in a block 
            i += 1
            for t in range(len(subject_epoch)): 
                # extract epoch  
                epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
                epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))
                
                # get features for epoch 
                epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
                epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)


                

                # fill data frame 
                ''' 
                features.loc[t] = [
                    subject, 
                    int(block),
                    t + 1,
                    subject_epoch[t].metadata['True_activation'].iloc[0],
                    subject_epoch.metadata.iloc[t]["Nb_Zygo"],
                    subject_epoch.metadata.iloc[t]["Nb_Corr"],
                    epoch_wl_zygo,
                    epoch_var_zygo, 
                    epoch_rms_zygo, 
                    epoch_wl_corr,
                    epoch_var_corr, 
                    epoch_rms_corr, 
     
                ]
                features_results_mat.append(features)
                '''


print(f"{i} Subjects & Naps Processed")

### Plot a single subject and epoch

In [14]:
# define subject 
subject = 'RL09PC'
block = '01' #nap number
epoch = 3 
subject_epoch, _ = pre_process_subjets(subject,block)

In [15]:
# get features 
epoch_zygo = np.squeeze(subject_epoch[epoch].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[epoch].get_data(picks=['Corr']))

# get features for epoch 
epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

In [23]:
fig = plot_epoch(epoch_corr,epoch_zygo,epoch_var_corr,epoch_wl_corr,epoch_rms_corr,
           epoch_var_zygo,epoch_wl_zygo,epoch_rms_zygo)

title = fig.suptitle(
        f"Facial EMG Response Following Stimulus | subject: {subject} | block: {block} | epoch: {epoch}",
        fontsize=14,
    )
plt.show()

KeyboardInterrupt: 

### Loop through all epochs of a single subject

In [ ]:
subject = 'RL09PC'
block = '01' #nap number
subject_epoch, _ = pre_process_subjets(subject,block) # preprocess subject to get epochs in a block 
current_index = 0 

In [41]:
# extract epoch  
epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

# get features for epoch 
epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

fig = plot_epoch(epoch_corr,epoch_zygo,epoch_var_corr,epoch_wl_corr,epoch_rms_corr,
        epoch_var_zygo,epoch_wl_zygo,epoch_rms_zygo)

fig.canvas.mpl_connect('key_press_event', on_key)


title = fig.suptitle(
    f"subject: {subject} | block: {block} | epoch: {current_index}",
    fontsize=14,
)
plt.show()